In [12]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)


import dagshub
import mlflow

dagshub.init(
    repo_owner="sergi.gonzalez.martos",
    repo_name="MLOps_AlgoRhythm",
    mlflow=True
)

mlflow.set_experiment("spotify-song-popularity")

# ----------------------------- CONFIGURATION -----------------------------
DATA_PATH = "../data/processed/dataset_transformed.csv"
TARGET_COL = "popularity"
# --------------------------------------------------------------------------


Initialized MLflow to track repo "sergi.gonzalez.martos/MLOps_AlgoRhythm"

Repository sergi.gonzalez.martos/MLOps_AlgoRhythm initialized!

In [13]:

df = pd.read_csv(DATA_PATH)

df['explicit'] = df['explicit'].astype(int)


print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

Loaded dataset: 108,646 rows x 129 columns


,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,genre_acoustic,genre_afrobeat,genre_alt-rock,genre_alternative,genre_ambient,genre_anime,genre_black-metal,genre_bluegrass,genre_blues,genre_brazil,...,genre_reggaeton,genre_rock,genre_rock-n-roll,genre_rockabilly,genre_romance,genre_sad,genre_salsa,genre_samba,genre_sertanejo,genre_show-tunes,genre_singer-songwriter,genre_ska,genre_sleep,genre_songwriter,genre_soul,genre_spanish,genre_study,genre_swedish,genre_synth-pop,genre_tango,genre_techno,genre_trance,genre_trip-hop,genre_turkish,genre_world-music
0,73,230666,0,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,55,149610,0,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,57,210826,0,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,71,201933,0,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,82,198853,0,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [14]:
col_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
col_booleanas = df.select_dtypes(include=['bool']).columns.tolist()
col_texto = df.select_dtypes(include=['object', 'category']).columns.tolist()

generos_one_hot = [col for col in df.columns if col.startswith('genre_')]
metricas_audio = [col for col in df.columns if col not in generos_one_hot and col != 'explicit']

print(f"numeric: {len(col_numericas)}")
print(f"bool: {len(col_booleanas)}")
print(f"text: {len(col_texto)}")
print(f"genre one hot encoded: {len(generos_one_hot)}")

resumen_columnas = pd.DataFrame({
    'Tipo_Dato': df.dtypes,
    'Valores_Nulos': df.isnull().sum(),
    'Valores_Unicos': df.nunique()
})

resumen_columnas.head(20)


numeric: 129
bool: 0
text: 0
genre one hot encoded: 114


,Tipo_Dato,Valores_Nulos,Valores_Unicos
popularity,int64,0,101
duration_ms,int64,0,47671
explicit,int64,0,2
danceability,float64,0,1087
energy,float64,0,1559
key,int64,0,12
loudness,float64,0,17233
mode,int64,0,2
speechiness,float64,0,1081
acousticness,float64,0,5057


## All audio features

In [15]:
audio_features = [
    'duration_ms', 'danceability', 'energy', 'key', 'loudness',
    'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature', 'explicit'
]
target = 'popularity'
X = df[audio_features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Linear Regression

In [16]:

# Scale data and Linear Regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# Train and predict
with mlflow.start_run(run_name="linear-regression"):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Log parameters
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("features", ", ".join(audio_features))
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.2f}")

MAE: 18.48
RMSE: 22.12
R2 Score: 0.03
🏃 View run linear-regression at: https://dagshub.com/sergi.gonzalez.martos/MLOps_AlgoRhythm.mlflow/#/experiments/0/runs/9a455f351f6a4904b9cfb923163d856c
🧪 View experiment at: https://dagshub.com/sergi.gonzalez.martos/MLOps_AlgoRhythm.mlflow/#/experiments/0


In [17]:
print("y_test:")
print(y_test.describe())

print("\ny_pred:")
print(pd.Series(y_pred).describe())

print("\nFirst 10 predictions:")
print(pd.DataFrame({
    "actual": y_test.iloc[:10].values,
    "predicted": y_pred[:10]
}))

y_test:
count    21730.000000
mean        33.295444
std         22.405097
min          0.000000
25%         17.000000
50%         35.000000
75%         50.000000
max         98.000000
Name: popularity, dtype: float64

y_pred:
count    21730.000000
mean        33.385905
std          3.788190
min         12.410595
25%         31.354318
50%         33.752703
75%         35.997922
max         46.422270
dtype: float64

First 10 predictions:
   actual  predicted
0      45  37.453266
1       3  32.847706
2      40  36.305810
3       0  23.077881
4      27  32.999303
5       5  32.533174
6      56  35.971713
7       0  35.403510
8      74  41.113775
9      28  34.402583


### Random Forest Regressor

In [18]:
# Train and predict
with mlflow.start_run(run_name="rfr"):
    rf_model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    y_pred_rf = rf_model.predict(X_test)

    mae_rf = mean_absolute_error(y_test, y_pred_rf)
    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    r2_rf = r2_score(y_test, y_pred_rf)

# Log parameters
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("features", ", ".join(audio_features))
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("mae", mae_rf)
    mlflow.log_metric("rmse", rmse_rf)
    mlflow.log_metric("r2", r2_rf)
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.2f}")



MAE: 18.48
RMSE: 22.12
R2 Score: 0.03
🏃 View run rfr at: https://dagshub.com/sergi.gonzalez.martos/MLOps_AlgoRhythm.mlflow/#/experiments/0/runs/2154e1afab024066bdb83b2edfb4fa50
🧪 View experiment at: https://dagshub.com/sergi.gonzalez.martos/MLOps_AlgoRhythm.mlflow/#/experiments/0


In [19]:
print("Actual mean:", y_test.mean())
print("RF prediction mean:", y_pred_rf.mean())

print("\nFirst 10 predictions:")
print(pd.DataFrame({
    "actual": y_test.iloc[:10].values,
    "predicted": y_pred_rf[:10]
}))

Actual mean: 33.29544408651633
RF prediction mean: 33.30922698736606

First 10 predictions:
   actual  predicted
0      45  37.585000
1       3   1.179905
2      40  41.380000
3       0  11.810000
4      27  44.927500
5       5  33.356667
6      56  32.540000
7       0  12.370000
8      74  66.070000
9      28  36.835000


### Gradient Boosting Regressor

In [20]:
# Gradient Boosting
# Train and predict
with mlflow.start_run(run_name="gradient-boosting"):
    gb_model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    gb_model.fit(X_train, y_train)

    # Predict
    y_pred_gb = gb_model.predict(X_test)

    # Gradient Boosting metrics
    mae_gb = mean_absolute_error(y_test, y_pred_gb)
    rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
    r2_gb = r2_score(y_test, y_pred_gb)

    # Log parameters
    mlflow.log_param("model", "GradientBoostingRegressor")
    mlflow.log_param("features", ", ".join(audio_features))
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("mae", mae_gb)
    mlflow.log_metric("rmse", rmse_gb)
    mlflow.log_metric("r2", r2_gb)

    print(f"MAE:  {mae_gb:.2f}")
    print(f"RMSE: {rmse_gb:.2f}")
    print(f"R²:   {r2_gb:.2f}")


MAE:  17.66
RMSE: 21.39
R²:   0.09
🏃 View run gradient-boosting at: https://dagshub.com/sergi.gonzalez.martos/MLOps_AlgoRhythm.mlflow/#/experiments/0/runs/1a43790b9fc44eb39c31ccbf1597a43f
🧪 View experiment at: https://dagshub.com/sergi.gonzalez.martos/MLOps_AlgoRhythm.mlflow/#/experiments/0


### results

In [21]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'Gradient Boosting'
    ],
    'MAE': [
        mae,
        mae_rf,
        mae_gb
    ],
    'RMSE': [
        rmse,
        rmse_rf,
        rmse_gb
    ],
    'R2': [
        r2,
        r2_rf,
        r2_gb
    ]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,18.480155,22.116763,0.025528
1,Random Forest,10.650316,15.042763,0.549202
2,Gradient Boosting,17.656494,21.387601,0.088723


In [22]:
## Add graphs and track them in mlflow, save final model as well